# 03 — Attaquer le modèle

**IA-01 · Jour 4 · Hexalp**

Cet après-midi, vous changez de camp. Vous êtes l'attaquant, et vous savez qu'Hexalp surveille son bastion avec votre modèle. Vous allez :

| Partie | Ce que fait l'attaquant | Nom du risque |
|---|---|---|
| A | Deviner les règles du modèle en l'interrogeant | Extraction, divulgation |
| B | Passer sous le radar | Contournement (évasion) |
| C | Profiter d'une liste blanche | Contournement |
| D | Fausser l'apprentissage | Empoisonnement |

Remplacez chaque `___`. Chaque ✍️ appelle une réponse dans votre carnet de bord.

⚠️ Tout se passe dans ce notebook, sur des données fictives. Ces techniques ne s'emploient jamais contre un système réel sans l'autorisation écrite de son propriétaire.

---
## Mise en place : le modèle d'Hexalp

La cellule suivante reconstruit exactement le modèle du J3 : même préparation, mêmes caractéristiques, même arbre, même Isolation Forest.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import IsolationForest
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, accuracy_score

import numpy as np

GRAINE = 42
CARACTERISTIQUES = ["echecs_auth", "succes_auth", "comptes_distincts", "duree_session_min",
                    "octets_sortants_mo", "heure", "externe"]

septembre = pd.read_csv("hexalp_ssh_sept2026.csv")
septembre["octets_sortants_mo"] = septembre["octets_sortants_mo"].fillna(septembre["octets_sortants_mo"].median())
septembre["externe"] = (septembre["origine"] == "externe").astype(int)
X = septembre[CARACTERISTIQUES]
y = (septembre["etiquette"] == "attaque").astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=GRAINE)
arbre = DecisionTreeClassifier(max_depth=3, random_state=GRAINE).fit(X_train, y_train)
iforest = IsolationForest(contamination=0.05, random_state=GRAINE).fit(X)
print("Modèle d'Hexalp prêt.")

In [ ]:
# Vous n'avez pas besoin de comprendre le détail de cette cellule.
# Elle définit verdict() : on décrit une heure d'activité, le modèle d'Hexalp répond.

def verdict(echecs_auth=0, succes_auth=1, comptes_distincts=1, duree_session_min=30,
            octets_sortants_mo=1, heure=14, externe=0, afficher=True):
    obs = pd.DataFrame([dict(echecs_auth=echecs_auth, succes_auth=succes_auth, comptes_distincts=comptes_distincts,
                             duree_session_min=duree_session_min, octets_sortants_mo=octets_sortants_mo,
                             heure=heure, externe=externe)])[CARACTERISTIQUES]
    avis_arbre = int(arbre.predict(obs)[0])
    score = float(iforest.decision_function(obs)[0])
    alerte = avis_arbre == 1 or score < 0
    if afficher:
        print(f"Arbre : {'ATTAQUE' if avis_arbre else 'normal':8s} | Isolation Forest : {score:+.3f} "
              f"({'anomalie' if score < 0 else 'rien'})  →  {'🚨 ALERTE' if alerte else '✅ aucune alerte'}")
    return alerte

# Exemple : un administrateur interne qui travaille à 10 h
verdict(succes_auth=3, duree_session_min=40, heure=10)

---
## Partie A — Interroger le modèle

L'attaquant ne voit pas le modèle. Mais imaginons qu'il puisse savoir s'il a été repéré : un compte qui se verrouille, une IP qui ne répond plus, un appel du SOC. Chaque essai devient alors une **question posée au modèle**.

La fonction `sonde()` simule ce que voit l'attaquant : une heure de password spraying depuis Internet (10 échecs, aucun succès), dont il choisit le nombre de comptes visés et l'heure. Il ne voit que la réponse : repéré ou pas.

In [ ]:
nb_requetes = 0

def sonde(comptes_distincts, heure):
    global nb_requetes
    nb_requetes = nb_requetes + 1
    alerte = verdict(echecs_auth=10, succes_auth=0, comptes_distincts=comptes_distincts,
                     duree_session_min=0, octets_sortants_mo=0, heure=heure, externe=1, afficher=False)
    print(f"Requête n°{nb_requetes} : {comptes_distincts} comptes à {heure} h → {'REPÉRÉ' if alerte else 'pas repéré'}")

**Mission A.** À 14 h, à partir de combien de comptes visés l'attaquant est-il repéré ? Les comptes vont de 1 à 64. Trouvez la réponse **en un minimum de requêtes** : modifiez la valeur et relancez la cellule autant de fois que nécessaire.

In [ ]:
sonde(comptes_distincts=___, heure=14)

**Mission A bis.** Avec 2 comptes visés, à partir de quelle heure du matin l'attaquant n'est-il plus repéré ?

In [ ]:
sonde(comptes_distincts=2, heure=___)

In [ ]:
# Comparez avec ce que verrait quelqu'un qui a accès au script ou à ses journaux
print(export_text(arbre, feature_names=CARACTERISTIQUES))

✍️ **A1.** Quels seuils avez-vous trouvés ? En combien de requêtes ? Quelle méthode avez-vous suivie ?

✍️ **A2.** Combien de requêtes faudrait-il, au minimum, si les comptes allaient de 1 à 1 000 ? Cherchez le terme **recherche dichotomique**.

✍️ **A3.** Un attaquant qui lit les règles de l'arbre n'a même pas besoin de sonder. Où ces règles peuvent-elles se trouver chez Hexalp ? Pensez au script, au dépôt de code, au fichier d'alertes, aux scores, au partage réseau.

---
## Partie B — Passer sous le radar

### B1. Le password spraying

L'attaquant veut tester le plus de mots de passe possible sur les comptes d'Hexalp, depuis une seule adresse IP, **sans déclencher d'alerte**. Proposez un plan : nombre d'échecs par heure, nombre de comptes visés, heure.

In [ ]:
# Votre plan, pour une heure
ECHECS = ___
COMPTES = ___
HEURE = ___

verdict(echecs_auth=ECHECS, succes_auth=0, comptes_distincts=COMPTES, duree_session_min=0,
        octets_sortants_mo=0, heure=HEURE, externe=1)

In [ ]:
# Combien d'essais par jour, si l'attaquant répète ce plan à chaque heure où il n'est pas repéré ?
heures_sans_alerte = [h for h in range(24)
                      if not verdict(echecs_auth=ECHECS, succes_auth=0, comptes_distincts=COMPTES, duree_session_min=0,
                                     octets_sortants_mo=0, heure=h, externe=1, afficher=False)]
print("Heures sans alerte :", heures_sans_alerte)
print("Essais par jour et par IP :", ECHECS * len(heures_sans_alerte))

✍️ **B1.** Quel est votre plan ? Combien de mots de passe l'attaquant peut-il tester par jour, avec une seule IP ? Et avec un botnet de 200 adresses ?

✍️ **B2.** Le modèle n'est pas seul : fail2ban bannit une IP après 5 échecs en 10 minutes. Votre plan y résiste-t-il ? Quel rythme maximal l'attaquant doit-il tenir ? Qu'en concluez-vous sur l'intérêt de garder la règle à côté du modèle ?

### B2. L'exfiltration

L'attaquant a volé des identifiants VPN. Il arrive avec une adresse interne et veut faire sortir une base de **5 000 Mo**. En une seule heure, à 23 h :

In [ ]:
verdict(succes_auth=1, comptes_distincts=1, duree_session_min=45, octets_sortants_mo=___, heure=23, externe=0)

In [ ]:
# Le volume maximal qui passe sans alerte, heure par heure
for heure in [1, 3, 10, 14, 23]:
    for volume in [500, 1000, 1500, 2000, 3000, 4000]:
        if verdict(succes_auth=1, comptes_distincts=1, duree_session_min=45, octets_sortants_mo=volume,
                   heure=heure, externe=0, afficher=False):
            print(f"{heure:2d} h : alerte à partir de {volume} Mo")
            break
    else:
        print(f"{heure:2d} h : aucune alerte, même à 4 000 Mo")

✍️ **B3.** Proposez un plan d'exfiltration des 5 000 Mo qui ne déclenche aucune alerte : combien d'heures, quel volume par heure, à quel moment ? Qu'est-ce que l'attaquant y perd ?

✍️ **B4.** Quelle caractéristique, absente du modèle, aurait gêné cet attaquant ? Pensez à ce que le modèle ne voit pas quand il regarde une seule heure.

---
## Partie C — La liste blanche

Ce matin, la sauvegarde nocturne `10.20.8.30` a encore produit des faux positifs. La solution qui vient tout de suite : l'ajouter à une **liste blanche**. Ses lignes ne seront plus jamais signalées.

In [ ]:
# Le modèle appliqué à octobre (préparation corrigée de ce matin)
octobre = pd.read_csv("hexalp_ssh_oct2026.csv")
octobre["octets_sortants_mo"] = octobre["octets_sortants_mo"].fillna(octobre["octets_sortants_ko"] / 1024)
octobre["octets_sortants_mo"] = octobre["octets_sortants_mo"].fillna(octobre["octets_sortants_mo"].median())
octobre["externe"] = (octobre["origine"] == "externe").astype(int)
X_o = octobre[CARACTERISTIQUES]
octobre["alerte"] = ((arbre.predict(X_o) == 1) | (iforest.predict(X_o) == -1)).astype(int)

LISTE_BLANCHE = [___]

dans_liste = octobre["ip_source"].isin(LISTE_BLANCHE)
print("Alertes avant la liste blanche :", octobre["alerte"].sum())
print("Alertes supprimées             :", octobre.loc[dans_liste, "alerte"].sum())
print("Alertes après la liste blanche :", octobre.loc[~dans_liste, "alerte"].sum())

In [ ]:
# Un attaquant a pris la main sur le serveur de sauvegarde. À 1 h, il fait sortir 9 000 Mo.
print("Ce que dirait le modèle :")
verdict(succes_auth=1, comptes_distincts=1, duree_session_min=55, octets_sortants_mo=9000, heure=1, externe=0)
print("Ce que verra l'analyste : rien, l'adresse est dans la liste blanche.")

✍️ **C1.** Combien d'alertes la liste blanche fait-elle gagner par semaine ? Combien de temps pour l'astreinte, à 10 minutes par alerte ?

✍️ **C2.** Que peut faire un attaquant qui connaît la liste blanche ? Qui, chez Hexalp, peut la modifier ?

✍️ **C3.** Proposez une règle plus étroite qu'une adresse entière : que pourriez-vous accepter de la sauvegarde, et à quelles conditions ?

---
## Partie D — Empoisonner l'apprentissage

### D1. Avec les étiquettes

Les données d'entraînement du modèle sont rangées sur un partage réseau. Tout le service informatique peut y écrire. Un attaquant qui a pris pied sur un poste y ajoute quelques **fausses lignes étiquetées `normal`**, qui ressemblent trait pour trait à l'attaque qu'il prépare : du spraying depuis Internet, en journée, sur 4 à 6 comptes.

Avant l'empoisonnement, le modèle repère cette attaque :

In [ ]:
attaque_preparee = dict(echecs_auth=6, succes_auth=0, comptes_distincts=5, duree_session_min=0,
                        octets_sortants_mo=0, heure=14, externe=1)
verdict(**attaque_preparee)

In [ ]:
N_LIGNES = ___      # le nombre de fausses lignes ajoutées par l'attaquant

rng = np.random.default_rng(GRAINE)
fausses = pd.DataFrame({
    "echecs_auth": rng.integers(4, 9, N_LIGNES), "succes_auth": 0, "comptes_distincts": rng.integers(4, 7, N_LIGNES),
    "duree_session_min": 0, "octets_sortants_mo": 0.0, "heure": rng.integers(9, 18, N_LIGNES), "externe": 1,
})[CARACTERISTIQUES]

X_empoisonne = pd.concat([X_train, fausses])
y_empoisonne = pd.concat([y_train, pd.Series(0, index=fausses.index)])
arbre_empoisonne = DecisionTreeClassifier(max_depth=3, random_state=GRAINE).fit(X_empoisonne, y_empoisonne)

obs = pd.DataFrame([attaque_preparee])[CARACTERISTIQUES]
print(f"{N_LIGNES} fausses lignes, soit {100 * N_LIGNES / len(X_empoisonne):.1f} % des données d'entraînement")
print("L'attaque préparée est classée :", "ATTAQUE" if arbre_empoisonne.predict(obs)[0] == 1 else "normal")
print()
print("Sur le jeu de test de septembre :")
print(classification_report(y_test, arbre_empoisonne.predict(X_test), target_names=["normal", "attaque"], digits=2))

Testez plusieurs valeurs de `N_LIGNES` : 0, 2, 4, 5, 10, 20.

✍️ **D1.** Combien de fausses lignes suffisent pour que l'attaque passe ? Quelle part des données d'entraînement cela représente-t-il ?

✍️ **D2.** Comparez le rapport du jeu de test avec et sans empoisonnement. L'équipe de supervision aurait-elle pu s'en apercevoir en regardant les métriques ?

### D2. Sans toucher aux étiquettes

L'Isolation Forest n'utilise aucune étiquette. Pour l'empoisonner, l'attaquant n'a rien à modifier : il lui suffit de **se comporter**. Pendant plusieurs nuits, avant son exfiltration, il fait sortir de gros volumes par le VPN. Ces nuits finissent dans les données d'entraînement, et le modèle apprend que c'est normal.

In [ ]:
N_NUITS = ___       # le nombre de nuits de « préparation »

rng = np.random.default_rng(GRAINE)
nuits = pd.DataFrame({
    "echecs_auth": rng.integers(0, 2, N_NUITS), "succes_auth": 1, "comptes_distincts": 1,
    "duree_session_min": rng.integers(30, 61, N_NUITS), "octets_sortants_mo": rng.uniform(3000, 9000, N_NUITS),
    "heure": rng.choice([3, 4, 5, 22, 23], N_NUITS), "externe": 0,
})[CARACTERISTIQUES]

iforest_empoisonne = IsolationForest(contamination=0.05, random_state=GRAINE).fit(pd.concat([X, nuits]))

# Les 9 exfiltrations par VPN d'octobre
exfiltrations = octobre[(octobre["etiquette"] == "attaque") & octobre["ip_source"].str.startswith("10.20.250.")]
print("Exfiltrations VPN repérées par l'Isolation Forest d'origine    :", (iforest.predict(exfiltrations[CARACTERISTIQUES]) == -1).sum(), "sur", len(exfiltrations))
print("Exfiltrations VPN repérées par l'Isolation Forest empoisonné :", (iforest_empoisonne.predict(exfiltrations[CARACTERISTIQUES]) == -1).sum(), "sur", len(exfiltrations))

Testez `N_NUITS` = 0, 10, 20, 40, 80.

✍️ **D3.** À partir de combien de nuits de préparation des exfiltrations commencent-elles à passer ? Combien en passe-t-il après 80 nuits ? De quoi l'attaquant a-t-il besoin : d'un accès aux fichiers, aux étiquettes, ou seulement de temps ?

✍️ **D4.** Proposez trois mesures qui protègent les données d'entraînement et le modèle. Pensez aux droits d'accès, à l'intégrité des fichiers, à la revue des étiquettes et à ce qu'on vérifie avant de remplacer un modèle par un nouveau.

---
✅ **Fin du notebook.** Reportez vos réponses dans la partie « Résistance aux attaques » de votre rapport d'évaluation.